# Đánh giá Chất lượng Agentic RAG so với Naive RAG (150 Câu)\n
Notebook này gọi trực tiếp hàm xử lý `llm_agent.py` của Backend (có tính năng Rewriter cho phép tự suy luận thay đổi câu hỏi 1 lần).\n

## 1. Import Backend Agent và Khởi tạo\n

In [ ]:
import os
import sys
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

# Link môi trường để chạy hàm Backend
sys.path.append(os.path.abspath('..'))

from backend.llm_agent import run_agentic_rag, init_agents
from backend.vector_store import get_retriever
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style("whitegrid")

EVAL_DIR = "../data/evaluation"
CSV_CHECKPOINT = os.path.join(EVAL_DIR, "evaluation_05_agentic_rag.csv")

print("Khởi tạo Backend Agent (Load models & DB)...")
init_agents()
get_retriever()
judge_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print("✅ Hoàn tất khởi tạo.")\n

## 2. Nạp Bảng Câu Hỏi & Hàm\n

In [ ]:
with open("../data/evaluation/150questions.txt", "r", encoding="utf-8") as f:
    content = f.read().strip()
    if content.startswith("```json"): content = content[7:]
    if content.endswith("```"): content = content[:-3]
    test_dataset = json.loads(content)
print(f"✅ Đã tải {len(test_dataset)} câu hỏi test.")

from groq import RateLimitError

def log_retry_attempt(retry_state):
    print(f"   ⏳ [Rate Limit] Groq báo quá tải chặn quyền truy cập. Đang tự đợi {retry_state.next_action.sleep:.1f} giây để lách luật (Lần {retry_state.attempt_number})...")

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=60), 
    stop=stop_after_attempt(10), 
    retry=retry_if_exception_type((RateLimitError, Exception)),
    before_sleep=log_retry_attempt
)
def evaluate_answer(question: str, ground_truth: str, generated_answer: str):
    judge_prompt = ChatPromptTemplate.from_template('''Đánh giá từ 1-10 chất lượng trả lời AI. Định dạng JSON CHUẨN: {{"score": 8, "reasoning": "Chi tiết đúng"}}
Câu hỏi: {question}
Chuẩn: {ground_truth}
AI: {generated_answer}''')
    res = judge_llm.invoke(judge_prompt.format(question=question, ground_truth=ground_truth, generated_answer=generated_answer))
    import re
    match = re.search(r'\{.*?\}', res.content.replace('\n
', ''))
    if match: return json.loads(match.group(0))
    return {"score": 5, "reasoning": "Lỗi Parse"}
\n

## 3. Sinh Câu Trả Lời (Agentic RAG Engine)\n

In [ ]:
def check_done(q):
    if not os.path.exists(CSV_CHECKPOINT): return False
    df = pd.read_csv(CSV_CHECKPOINT, encoding='utf-8-sig')
    return len(df[(df['question'] == q) & (df['generated_answer'].notna())]) > 0

def save_csv(data):
    pd.DataFrame([data]).to_csv(CSV_CHECKPOINT, mode='a', header=not os.path.exists(CSV_CHECKPOINT), index=False, encoding='utf-8-sig')

completed = sum(1 for item in test_dataset if check_done(item["question"]))
print(f"📊 Đã hoàn thành trước đó: {completed}/{len(test_dataset)} câu. Các câu này sẽ được bỏ qua.")

pbar = tqdm(total=len(test_dataset), initial=completed, desc="Agentic Generating")

for idx, item in enumerate(test_dataset):
    if check_done(item["question"]): 
        pbar.update(1)
        continue
    
    try:
        start_time = time.time()
        print(f"\n
{'='*80}")
        print(f"🤖 Đang gửi yêu cầu sinh câu số {idx+1}: {item['question']}")
        
        # Chạy vòng lặp Agentic! (Ghi chú: Bản thân llm_agent của bạn không có hàm @retry, 
        # do đó bạn sẽ cần cẩn trọng nếu Backend bị Limit. Chúng ta xử lý ngoại lệ ở main try)
        ans, ctx, intent, retries = run_agentic_rag(item['question'], max_retries=1, is_thinking=True)
        latency = time.time() - start_time
        
        if retries > 0:
            print(f"   ⚠️ BOT TỰ SỬA CÂU HỎI (Retry: {retries} lần) để khôi phục Context!")
        print(f"✅ Hoàn thành! ({latency:.2f}s) - Phản hồi: {ans[:100]}...")
        
        save_csv({
            'question_id': item['id'], "question": item['question'], "ground_truth": item['expected_answer'], 
            "generated_answer": ans, "intent": intent, "retries_count": retries,
            "latency_seconds": round(latency, 2), "score": None, "reasoning": None
        })
    except Exception as e:
        print(f"❌ LỖI NGHIÊM TRỌNG TẠI CÂU {idx+1}: Bị chặn API Rate Limit tại Backend hoặc mất kế nối: {e}")
        save_csv({
            'question_id': item['id'], "question": item['question'], "ground_truth": item['expected_answer'], 
            "generated_answer": "ERROR", "intent": "ERROR", "retries_count": 0, "latency_seconds": 0, 
            "score": 0, "reasoning": str(e)
        })
    
    pbar.update(1)
    time.sleep(1)

pbar.close()
print("✅ HOÀN TẤT SINH CÂU TRẢ LỜI CHO AGENTIC.")\n

## 4. Chấm Điểm Bằng LLM-as-a-Judge\n

In [ ]:
df = pd.read_csv(CSV_CHECKPOINT, encoding='utf-8-sig')
need_eval = df[df['score'].isna() | df['score'].isnull()]
print(f"🔍 Có {len(need_eval)} câu cần chấm điểm.")

if len(need_eval) > 0:
    for idx, row in tqdm(need_eval.iterrows(), total=len(need_eval), desc="Tiến độ Chấm điểm"):
        if str(row['generated_answer']) == 'ERROR' or str(row['generated_answer']).strip() == '': 
            continue
        try:
            print(f"\n
⚖️ Đang chấm điểm câu số {row['question_id']}...")
            ev = evaluate_answer(row['question'], row['ground_truth'], row['generated_answer'])
            print(f"   => ĐIỂM ĐẠT: {ev.get('score', 5)}/10 | LÝ DO: {ev.get('reasoning', '')}")
            df.at[idx, 'score'] = ev.get('score', 5)
            df.at[idx, 'reasoning'] = ev.get('reasoning', '')
            df.to_csv(CSV_CHECKPOINT, index=False, encoding='utf-8-sig')
        except Exception as e:
            print(f"❌ LỖI CHẤM ĐIỂM CÂU {row['question_id']}: {e}")
        time.sleep(1)
    print("✅ ĐÃ CHẤM XONG TOÀN BỘ!")
else:
    print("✅ Tất cả câu trả lời đã được chấm điểm.")\n

## 5. Bảng So Sánh (Naive RAG vs Agentic RAG)\n

In [ ]:
df_05 = pd.read_csv(CSV_CHECKPOINT, encoding='utf-8-sig')
print("--- KẾT QUẢ AGENTIC RAG (Max 1 Retry) ---")
print(f"📉 Điểm Trung Bình: {df_05['score'].mean():.2f}/10")
print(f"⏱️ Thời gian TB: {df_05['latency_seconds'].mean():.2f} giây")

retry_cases = df_05[df_05['retries_count'] > 0]
print(f"🔄 Có {len(retry_cases)} câu hỏi phải tự động viết lại (Retry) ít nhất 1 lần.")
if len(retry_cases) > 0:
    print(f"📉 Điểm TB của đám tự viết lại: {retry_cases['score'].mean():.2f}/10")

if os.path.exists("../data/evaluation/evaluation_04_naive_rag.csv"):
    df_04 = pd.read_csv("../data/evaluation/evaluation_04_naive_rag.csv", encoding='utf-8-sig')
    compare_df = pd.DataFrame({
        "Mô hình": ["Naive RAG (04)", "Agentic RAG (05)"],
        "Điểm TB": [df_04['score'].mean(), df_05['score'].mean()],
        "Thời gian TB (s)": [df_04['latency_seconds'].mean(), df_05['latency_seconds'].mean()]
    })
    print("📊 SO SÁNH 2 LUỒNG TOÀN DIỆN:")
    print(compare_df)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sns.barplot(data=compare_df, x="Mô hình", y="Điểm TB", ax=axes[0], palette=["#3498db", "#e74c3c"])
    axes[0].set_title("So Sánh Điểm Số Khảo Sát (Cao = Tốt)", fontsize=14, fontweight='bold')
    axes[0].set_ylim(0, 10)
    for i, v in enumerate(compare_df["Điểm TB"]):
        axes[0].text(i, v + 0.2, f"{v:.2f}", ha="center", fontweight="bold")
    
    sns.barplot(data=compare_df, x="Mô hình", y="Thời gian TB (s)", ax=axes[1], palette=["#3498db", "#e74c3c"])
    axes[1].set_title("So Sánh Độ Trễ Phản Hồi (Thấp = Tốt)", fontsize=14, fontweight='bold')
    for i, v in enumerate(compare_df["Thời gian TB (s)"]):
        axes[1].text(i, v + 0.1, f"{v:.2f}s", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Vui lòng chạy xong File 04_Evaluation_Single_Model.ipynb trước để có Data vẽ biểu đồ so sánh.")